# データの読み込み

In [1]:
import subprocess
import json
import pandas as pd

In [2]:
cmd = [
    "wrangler", "d1", "execute", "citrus-db",
    "--remote",
    "--command", "SELECT * FROM user_logs;",
    "--json"
]

result = subprocess.run(cmd, capture_output=True, text=True)

data = json.loads(result.stdout)

# rows部分をDataFrame化
df = pd.DataFrame(data[0]["results"])


In [3]:
df.head()

,id,ts,session_id,input_json,result,user_id,1_a,1_r,1_s,2_a,2_r,2_s,3_a,3_r,3_s
0,3924,2026-04-17T12:32:45.639306+00:00,2133a8c1-fdee-4eec-bc36-d787a2744bc2,"{""brix"":4,""acid"":3,""bitterness"":2,""aroma"":3,""m...","[{""id"":45,""rank"":1},{""id"":49,""rank"":2},{""id"":5...",U335598cb5f88f6bc9610f503a86e6562,1,0,0,0,0,0,0,0,0
1,3925,2026-04-17T12:47:43.734034+00:00,c816eafb-19b7-40ae-9641-e079bb14c8b2,"{""brix"":3,""acid"":5,""bitterness"":2,""aroma"":5,""m...","[{""id"":1,""rank"":1},{""id"":8,""rank"":2},{""id"":17,...",U0687c215cb0e904e9e857fe77fcb2bfe,0,0,0,0,0,0,1,0,0
2,3926,2026-04-17T13:04:13.059754+00:00,b7ad8fae-bcd6-4625-ba8f-85a9f03d2529,"{""brix"":2,""acid"":6,""bitterness"":1,""aroma"":4,""m...","[{""id"":17,""rank"":1},{""id"":1,""rank"":2},{""id"":43...",Uaddb2a3cdee936b3775f4d8dc049f284,0,0,0,0,0,0,0,0,0
3,3927,2026-04-17T13:04:35.549727+00:00,eaea2a4e-a4f6-4736-9ad2-9aa262347ed7,"{""brix"":2,""acid"":4,""bitterness"":1,""aroma"":4,""m...","[{""id"":1,""rank"":1},{""id"":49,""rank"":2},{""id"":8,...",Uaddb2a3cdee936b3775f4d8dc049f284,0,0,0,0,0,0,0,0,0
4,3928,2026-04-17T13:05:09.322420+00:00,13952890-2874-4988-bc88-c04e1db091e8,"{""brix"":2,""acid"":4,""bitterness"":1,""aroma"":4,""m...","[{""id"":1,""rank"":1},{""id"":2,""rank"":2},{""id"":8,""...",Uaddb2a3cdee936b3775f4d8dc049f284,0,0,0,0,0,0,0,0,0


In [6]:
print(df.shape)
print(df["user_id"].count())

(31, 15)
19


29列うちで、18名の識別可能なユーザと11名の非ログインユーザのデータ。

In [5]:
df.dtypes

id            int64
ts              str
session_id      str
input_json      str
result          str
user_id         str
1_a           int64
1_r           int64
1_s           int64
2_a           int64
2_r           int64
2_s           int64
3_a           int64
3_r           int64
3_s           int64
dtype: object

# 類似入力ベース推薦の流れ

---

### 1. データ整形

元データを以下のような縦持ち形式に変換する。

| user_id | item_id | brix | acid | aroma | ... | liked |
|---|---|---|---|---|---|---|
| U1 | 45 | 4 | 3 | 3 | ... | 1 |
| U1 | 49 | 4 | 3 | 3 | ... | 0 |

- `liked = 1`
  - `1_a`, `1_r`, `1_s` のいずれかが1
- `liked = 0`
  - それ以外

---

### 2. 新しい入力

新しいユーザ入力を特徴ベクトル化する。

$x_{\text{new}} = [brix, acid, bitterness, aroma, moisture, texture]$

例：

$x_{\text{new}}=[5,4,1,5,4,5]$

---

### 3. 過去入力との距離計算

各過去ログ \(x_i\) とのユークリッド距離を計算する。

$d_i=\|x_{\text{new}} - x_i\|$

---

### 4. 類似度へ変換

距離を類似度へ変換する。

$sim_i=\frac{1}{1+d_i}$

- 距離が小さいほど類似度が高い
- 類似度は \( 0 \sim 1\)

---

### 5. 類似ログ抽出

$score =sim_i \times liked$

$(0 \leq score \leq 1 )$

類似度上位 \(k\) 件を使用する。

---

# FM実装

In [7]:
import pandas as pd
import numpy as np
import json
from collections import defaultdict

In [8]:
# =========================
# 1. result と input_json を展開して縦持ちデータに変換
# =========================

def build_interaction_df(df):
    records = []
    for _, row in df.iterrows():
        # input_json の展開
        input_data = row["input_json"]
        if isinstance(input_data, str):
            input_data = json.loads(input_data)

        # result の展開
        result_data = row["result"]

        if isinstance(result_data, str):
            result_data = json.loads(result_data)

        for item in result_data:
            rank = item["rank"]
            item_id = item["id"]

            # その順位の feedback のどれかが1なら liked=1
            liked = int(
                row[f"{rank}_a"] == 1 or
                row[f"{rank}_r"] == 1 or
                row[f"{rank}_s"] == 1
            )
            record = {
                "user_id": row["user_id"],
                "session_id": row["session_id"],
                "item_id": item_id,
                "rank": rank,
                "liked": liked,
            }

            # brix, acid などを追加
            record.update(input_data)
            records.append(record)

    return pd.DataFrame(records)

interactions = build_interaction_df(df)
interactions.head()

# =========================
# 2. 類似度ベースでスコアリングして推薦
# =========================

def recommend_by_input_similarity(
    interactions,
    new_input,
    feature_cols=None,
    top_n=5,
    k_neighbors=None
):
    """
    interactions: build_interaction_df(df) で作った縦持ちデータ
    new_input: 新しい嗜好入力 dict
    feature_cols: 類似度計算に使う特徴量
    top_n: 推薦する件数
    k_neighbors: 近い過去ログを何件使うか。Noneなら全件使用
    """

    if feature_cols is None:
        feature_cols = ["brix", "acid", "bitterness", "aroma", "moisture", "texture"]

    data = interactions.copy()

    # 新しい入力ベクトル
    x_new = np.array([new_input[col] for col in feature_cols], dtype=float)

    # 過去入力ベクトル
    X = data[feature_cols].astype(float).values

    # ユークリッド距離
    distances = np.linalg.norm(X - x_new, axis=1)

    # 距離を類似度に変換
    # 距離0なら類似度1、距離が大きいほど小さくなる
    similarities = 1 / (1 + distances)

    data["similarity"] = similarities

    # 近いログだけ使う場合
    if k_neighbors is not None:
        data = data.sort_values("similarity", ascending=False).head(k_neighbors)

    # itemごとのスコア計算
    item_scores = []

    for item_id, group in data.groupby("item_id"):
        # 類似度加重 liked 平均
        numerator = (group["similarity"] * group["liked"]).sum()
        denominator = group["similarity"].sum()

        score = numerator / denominator if denominator > 0 else 0

        # 補助情報
        liked_count = group["liked"].sum()
        shown_count = len(group)

        item_scores.append({
            "item_id": item_id,
            "score": score,
            "liked_count": liked_count,
            "shown_count": shown_count,
            "weighted_liked_sum": numerator,
            "weighted_shown_sum": denominator,
        })

    score_df = pd.DataFrame(item_scores)

    # スコア順に並べる
    score_df = score_df.sort_values(
        ["score", "weighted_liked_sum", "liked_count"],
        ascending=False
    )

    return score_df.head(top_n)

In [10]:
new_input = {
    "brix": 5,
    "acid": 4,
    "bitterness": 1,
    "aroma": 5,
    "moisture": 4,
    "texture": 5
}

recommendations = recommend_by_input_similarity(
    interactions=interactions,
    new_input=new_input,
    top_n=3,
    k_neighbors=100
)

recommendations

,item_id,score,liked_count,shown_count,weighted_liked_sum,weighted_shown_sum
6,11,1.000000,1,1,0.240253,0.240253
21,41,0.536675,1,2,0.231662,0.431662
16,33,0.480964,1,2,0.231662,0.481662
